In [2]:
from functools import lru_cache
import datetime
import numpy as np
from visualisation import plotly_utils as pu
import plotly.express as px
import polars as pl
import plotly
import plotly.io as pioget

In [11]:
from options_data_utils import get_cleaned_options_data
tlt_call = get_cleaned_options_data('XLE', (2023,2024), 'CALL')
delta_thresholds_puts = {
    "very_deep_otm_put": (-0.10, 0.0),
    "deep_otm_put": (-0.20, -0.10),
    "otm_put": (-0.45, -0.35),
    "around_atm_put": (-0.55, -0.45),
    "itm_put": (-0.75, -0.65),
    "deep_itm_put": (-0.90, -0.80),
    "very_deep_itm_put": (-1.0, -0.90)
}


delta_thresholds_calls = {
    "{k}": (0.0, 0.10),
    "deep_otm_call": (0.10, 0.20),
    "otm_call": (0.35, 0.45),
    "around_atm_call": (0.45, 0.55),
    "itm_call": (0.65, 0.75),
    "deep_itm_call": (0.80, 0.90),
    "very_deep_itm_call": (0.90, 1.0)
}

def get_average_iv_for_delta_bucket(options_data: pl.DataFrame, delta_thresholds: dict, bucket: str):
  return options_data.group_by("trading_day").agg([
      pl.col("implied_vol")
        .filter(pl.col("delta").is_between(*delta_thresholds[bucket]))
        .mean()
        .alias(bucket + "_iv")
  ]).sort("trading_day")

[round(get_average_iv_for_delta_bucket(tlt_call, delta_thresholds_calls, k).null_count()[f'{k}_iv'][0]/len(tlt_call),3) for k in delta_thresholds_calls.keys()]


[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

0

In [4]:
tlt_call.filter_day('20190920').filter(pl.col("delta").is_between(*delta_thresholds_calls["around_atm_call"]))

ticker,timestamp,strike,expiration,delta,theta,vega,rho,epsilon,lambda,implied_vol,iv_error,underlying_price,price,trading_day,spread
str,datetime[μs],i32,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,f64
"""TLT""",2019-09-20 09:35:00,141000,2019-10-18,0.5135,-0.0401,15.5575,5.3898,-5.5502,34.6112,0.1317,0.0001,140.88,2.095,2019-09-20,0.05
"""TLT""",2019-09-20 09:40:00,141000,2019-10-18,0.5112,-0.0398,15.557,5.3656,-5.5236,34.9564,0.1307,0.0,140.85,2.06,2019-09-20,0.04
"""TLT""",2019-09-20 09:45:00,141000,2019-10-18,0.501,-0.0397,15.5488,5.2557,-5.4084,35.4244,0.1304,0.0001,140.72,1.995,2019-09-20,0.03
"""TLT""",2019-09-20 09:50:00,141000,2019-10-18,0.4962,-0.0395,15.5415,5.2047,-5.3544,35.779,0.1298,0.0004,140.66,1.955,2019-09-20,0.03
"""TLT""",2019-09-20 09:55:00,141000,2019-10-18,0.4939,-0.0397,15.5371,5.1797,-5.3292,35.6393,0.1307,-0.0003,140.63,1.95,2019-09-20,0.02
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""TLT""",2019-09-20 15:40:00,142000,2019-10-18,0.4966,-0.0418,15.651,5.2376,-5.3971,33.8351,0.1374,-0.0003,141.65,2.085,2019-09-20,0.01
"""TLT""",2019-09-20 15:45:00,142000,2019-10-18,0.5107,-0.0423,15.6669,5.3895,-5.5575,33.075,0.1384,0.0001,141.84,2.19,2019-09-20,0.02
"""TLT""",2019-09-20 15:50:00,142000,2019-10-18,0.5173,-0.0425,15.6677,5.4601,-5.6327,32.6338,0.1392,0.0,141.93,2.255,2019-09-20,0.03


In [ ]:
os.environ['LOG_FILE_NAME'] = 'logs/test.log'
os.environ['INTERVAL'] = '300000'
os.environ['TYPE'] = 'C'
os.environ['FOLDER'] = 'CALL'
os.environ['USE_CACHE'] = 'False'

In [ ]:
from fetch_important_tickers import *
import datetime
trading_day = datetime.date(2019, 9, 20)
expiration_day = datetime.date(2019, 10, 18)
ticker = 'TLT'
strikes= subset_strikes(ticker=ticker, trading=trading_day, expiration=expiration_day)

In [ ]:
strikes['strike'].unique().to_list()

In [ ]:
def fetch (start_expirations_strike):      
    greeks_schema_dict = {
    'ms_of_day': pl.Int64,           # Milliseconds of the day, integer
    'bid': pl.Float64,               # Bid price, float
    'ask': pl.Float64,               # Ask price, float
    'delta': pl.Float64,             # Delta, float
    'theta': pl.Float64,             # Theta, float
    'vega': pl.Float64,              # Vega, float
    'rho': pl.Float64,               # Rho, float
    'epsilon': pl.Float64,           # Epsilon, float
    'lambda': pl.Float64,            # Lambda, float
    'implied_vol': pl.Float64,       # Implied volatility, float
    'iv_error': pl.Float64,          # Implied volatility error, float
    'ms_of_day2': pl.Int64,          # Milliseconds of the day (second instance), integer
    'underlying_price': pl.Float64,  # Underlying asset price, float
    'date': pl.String                  # Date, string
}

    trading_day, expiration, strike, ticker = start_expirations_strike
    trading_day = stringify_date(trading_day)
    expiration = stringify_date(expiration)
    url = f"http://127.0.0.1:25510/v2/hist/option/greeks?root={ticker}&exp={expiration}&strike={strike}&right=C&start_date={trading_day}&end_date={expiration}&ivl={os.environ['INTERVAL']}"
    try:
        response = requests.get(url)
        if response.status_code != 200:
            
            print(f"API returned a status code {response.status_code} for {ticker} with expiration {expiration} for trading day {trading_day}")
            print(f"RogueURL: {url}")
            df = pl.DataFrame(None, schema=greeks_schema_dict).with_columns([
            pl.lit(expiration).alias('expiration'),
            pl.lit(strike).alias('strike'),
            pl.lit(ticker).alias('ticker'),
            ])
            return df
        else:
            df =pl.DataFrame(response.json()['response'], schema=greeks_schema_dict, orient='row').with_columns([
            pl.lit(expiration).alias('expiration'),
            pl.lit(strike).alias('strike'),
            pl.lit(ticker).alias('ticker'),
        ]) 
    except Exception as e:
        print(f"Failed to fetch data for {ticker} with expiration {expiration}, strike {strike} for trading day {trading_day} with error {e}")
        # Handle empty dataframes (just in case)
        df = pl.DataFrame(None, schema=greeks_schema_dict).with_columns([
            pl.lit(expiration).alias('expiration'),
            pl.lit(strike).alias('strike'),
            pl.lit(ticker).alias('ticker'),
        ]) 
    return df

In [ ]:
paired_expirations_and_strike[25]

In [ ]:
relevant_data = get_relevant_expirations_and_strikes('TLT').select(['trading_day', 'expiration_day', 'strike','ticker'])
paired_expirations_and_strike = [row for row in relevant_data.iter_rows()]
test = fetch(paired_expirations_and_strike[25])

In [ ]:
relevant_data.filter(pl.col('trading_day').dt.year() < 2020)

In [ ]:
trading_day = datetime.date(2019, 9, 20)
expiration_day = datetime.date(2019, 10, 18)
relevant_data.filter(pl.col('trading_day').dt.year().is_between(2019, 2020)).filter(pl.col('trading_day') ==trading_day)
# relevant_data.filter(pl.col('trading_day').dt.year().is_between(2021, 2022))
# relevant_data.filter(pl.col('trading_day').dt.year().is_between(2023, 2024))


In [ ]:
800+1150+950

In [ ]:
tlt_call.filter_day('20190920')['strike'].unique()

In [ ]:
tlt_call.group_by("trading_day").agg([
      pl.col("strike").unique().count().alias("unique_strikes"),
     ]).sort("trading_day")

In [ ]:
#  I have polars dataframe with delta, IV and trading_day column. 
#  Give me a polars code that will create a new column called delta_bucket on the basis of these values

In [ ]:
#  I have the undelying price and multiple strike prices for a given expiry date. I want to filter ATM options. However 

In [5]:
delta_thresholds_puts = {
    "very_deep_otm_put": (-0.10, 0.0),
    "deep_otm_put": (-0.20, -0.10),
    "otm_put": (-0.45, -0.35),
    "around_atm_put": (-0.55, -0.45),
    "itm_put": (-0.75, -0.65),
    "deep_itm_put": (-0.90, -0.80),
    "very_deep_itm_put": (-1.0, -0.90)
}
def get_average_iv_for_delta_bucket(options_data: pl.DataFrame, delta_thresholds: dict, bucket: str):
  return options_data.group_by("trading_day").agg([
      pl.col("implied_vol")
        .filter(pl.col("delta").is_between(*delta_thresholds[bucket]))
        .mean()
        .alias(bucket + "_iv")
  ]).sort("trading_day")

In [ ]:
print(data_dia.filter_day('20190920')['underlying_price'].mean()*1000)
data_dia.filter_day('20190920')['strike'].describe()

In [ ]:
def find_till (data, delta):
    epsilon_search = [epsilon/100 for epsilon in range(1, 15)]
    data_filtered = pl.DataFrame(schema=data.schema)
    while (not len(data_filtered)) and epsilon_search:
        epsilon = epsilon_search.pop(0)                
        data_filtered = data.filter_delta(delta, epsilon)
    if not len(data_filtered):
        return None
    return data_filtered

In [ ]:
395*0.7

In [ ]:
data_alny.unique('trading_day')

In [ ]:
# data_alny.group_by('trading_day').agg(pl.col('delta').is_between(0.45, 0.55).mean())
data_dia.group_by("trading_day").agg([
    pl.col("delta")
      .filter((pl.col("delta").is_between(-0.26,-0.24)))
      .mean()
      .alias("average_delta")
]).sort("trading_day").filter(pl.col("average_delta").is_null())
# .null_count()


In [ ]:
import numpy as np

def make_list_sparse(dense_list, target_size):
    if target_size >= len(dense_list):
        return dense_list  # If target size is greater than or equal to the length of the list, return original list
    
    # Calculate evenly spaced indices based on target_size
    indices = np.linspace(0, len(dense_list) - 1, target_size, dtype=int)
    
    # Use the indices to select the elements from the dense list
    sparse_list = [dense_list[i] for i in indices]
    
    return sparse_list

# Example usage
dense_list = [
    250000, 255000, 260000, 265000, 270000, 275000, 280000, 285000, 290000, 295000,
    300000, 305000, 310000, 315000, 316000, 317000, 318000, 319000, 320000, 321000,
    322000, 323000, 324000, 325000, 326000, 327000, 328000, 329000, 330000, 331000,
    332000, 333000, 334000, 335000, 336000, 337000, 338000, 339000, 340000, 341000,
    342000, 343000, 344000, 345000, 346000, 347000, 348000, 349000, 350000, 351000,
    352000, 353000, 354000, 355000, 356000, 357000, 358000, 359000, 360000, 361000,
    362000, 363000, 364000, 365000, 366000, 367000, 368000, 369000, 370000, 371000,
    372000, 373000, 374000, 375000, 376000, 377000, 378000, 379000, 380000, 381000,
    382000, 382500, 383000, 384000, 385000, 386000, 387000, 387500, 388000, 389000,
    390000, 391000, 392000, 392500, 393000, 394000, 395000, 396000, 397000, 397500,
    398000, 399000, 400000, 401000, 402000, 402500, 403000, 404000, 405000, 406000,
    407000, 407500, 408000, 409000, 410000, 411000, 412000, 413000, 414000, 415000,
    416000, 417000, 418000, 419000, 420000, 421000, 422000, 423000, 424000, 425000,
    426000, 427000, 428000, 429000, 430000, 431000, 432000, 433000, 434000, 435000,
    436000, 437000, 438000, 439000, 440000, 441000, 442000, 443000, 444000, 445000,
    446000, 447000, 448000, 449000, 450000, 451000, 452000, 455000, 460000, 465000,
    470000, 475000, 480000, 485000, 490000, 495000, 500000
]

sparse_list = make_list_sparse(dense_list, 15)  # Example to reduce the list to 50 elements
print(sparse_list)


In [ ]:
data.filter_day('20190923').filter(pl.col('delta').is_between(0.20,0.30))['delta'].describe()
# filter_delta(0.25, epsilon=0.2)

In [ ]:
def check_existance_of_deltas (ticker, type, delta, epsilon=0.01):
    data = get_cleaned_options_data(ticker, type=type)
    trading
    deltas = data.filter_day('20190930').filter_delta(0.25, epsilon=0.01)
if not(len(deltas)):
    with open('deltas.txt', 'w') as f:
        f.write(str(deltas))

In [ ]:
data

In [ ]:
data_put = get_cleaned_options_data('XLE', type='PUT')
data_put.filter_day('20200131')['delta'].describe()

In [ ]:
data.filter_day('20190930').filter_delta(0.5, epsilon=0.001)

In [ ]:
data_put.filter_day('20190930').filter_delta(-0.5, epsilon=0.001)

In [ ]:
0.222

In [ ]:
data = get_cleaned_options_data('AAPL', type='PUT')


In [ ]:
data.filter_day('20190930')

In [ ]:
data


In [ ]:
from deb import get_relevant_expirations_and_strikes
get_relevant_expirations_and_strikes('AAPL').write_csv('test.csv')

In [ ]:
from data_fetcher import get_paired_expiration_and_strike_for_all_available_expirations_as_polars, subset_within_20_percent, append_log, get_spot_for_trading_day

import polars as pl
import os
import numpy as np
os.environ['LOG_FILE_NAME'] = 'logs/calldata.log'
os.environ['INTERVAL'] = '300000'
os.environ['TYPE'] = 'P'
os.environ['FOLDER'] = 'PUT'
os.environ['USE_CACHE'] = 'False'
import datetime
trading = datetime.datetime(2019, 9, 23)
expiration = datetime.datetime(2019, 10, 18)
ticker='AAPL'

def subset_within_20_percent(arr, num): 
    print(arr)
    append_log(f"Max: {arr.min()}, Median: {np.median(arr)}, Max: {arr.max()}, N: {len(arr)}", level='INFO', log_file="strikes.log")
    arr = np.array(arr)
    lower_bound = num * 0.7
    upper_bound = num * 1.3
    subset = arr[(arr >= lower_bound) & (arr <= upper_bound)]  
    if len(subset) > 15:
        subset = make_list_sparse(subset.tolist(), 15)  #
        subset_numpy = np.array(subset)
        append_log(f"Max: {subset_numpy.min()}, Median: {np.median(subset_numpy)}, Max: {subset_numpy.max()}, N: {len(subset_numpy)}", level='INFO', log_file="log/strikes.log")
        return subset
    else:
        return subset


def subset_strikes (ticker, trading, expiration, subset_strikes_func=subset_within_20_percent):
    # moneyness_array = get_moneyness_range(9, 11, 0.5)
    df = get_paired_expiration_and_strike_for_all_available_expirations_as_polars(ticker)
    spot_price = get_spot_for_trading_day(ticker, trading)
    if not len(df):
        return pl.DataFrame
    trading_expiration_pair = df.filter((pl.col('trading_day') == trading) & (pl.col('expiration_day') == expiration) & (pl.col('ticker') == ticker))

    nearest_strikes = subset_strikes_func(trading_expiration_pair['strike'].to_numpy(), spot_price*1000)
    print(nearest_strikes)
    # nearest_strikes = get_neighboring_strikes(ticker, trading, trading_expiration_pair['strike'])
    # nearest_strikes = [find_nearest_strike(ticker, moneyness/10, trading, .to_numpy()) for moneyness in moneyness_array]
    if len(nearest_strikes):
        subsetted_strikes = trading_expiration_pair.filter(pl.col('strike').is_in((nearest_strikes)))
        return subsetted_strikes
    else:
        append_log(f"No strikes found for {ticker} for trading day {trading} and expiration day {expiration}", level='ERROR', log_file=os.environ['LOG_FILE_NAME'])
        return pl.DataFrame(schema=trading_expiration_pair.schema)
subset_strikes('ALNY', trading, expiration)

In [ ]:
@lru_cache(maxsize=10)
def get_median_price_dataframe (filename='DATA/median_price_each_day.parquet'):
    return pl.read_parquet(filename)

In [ ]:

def get_paired_expirations (ticker):
    available_expirations = get_available_expirations(ticker)
    return pl.DataFrame(zip(available_expirations, available_expirations[1:]), schema=['current', 'next'])

get_paired_expirations('AAPL')

In [ ]:
@lru_cache(maxsize=128)
def get_unique_trading_days (ticker):
    return get_paired_expiration_and_strike_for_all_available_expirations_as_polars(ticker).filter(pl.col('trading_day').is_in(get_median_price_dataframe().filter(pl.col('ticker') == ticker)['date'])).select(['trading_day', 'expiration_day']).unique().sort('trading_day')

In [ ]:
def get_relevant_expirations_and_strikes (ticker):
    return pl.concat(subset_strikes(ticker, trading_day, expiration_day) for trading_day, expiration_day in get_unique_trading_days(ticker).iter_rows()).join(get_median_price_dataframe().filter(pl.col('ticker') == ticker).drop('ticker'), left_on='trading_day', right_on='date').sort('trading_day')


In [ ]:
get_unique_trading_days(ticker)

In [ ]:
def find_nearest_strike(ticker, moneyness, trading_day, strike_prices):
    spot_price = get_spot_for_trading_day(ticker, trading_day)
    strike_prices = np.array(strike_prices)/1000
    target_strike = moneyness * spot_price    
    differences = np.abs(np.array(strike_prices) - target_strike)    
    min_diff_idx = np.argmin(differences)
    min_diff = differences[min_diff_idx]
    candidates = [strike for i, strike in enumerate(strike_prices) if differences[i] == min_diff]
    return min(candidates)*1000

find_nearest_strike(ticker, 1.1, trading_day, strike_prices)

In [ ]:
trading = datetime.datetime(2019, 9, 20)
expiration = datetime.datetime(2019, 10, 18)
ticker='AAPL'
print(subset_strikes('AAPL', trading, expiration))
df = get_paired_expiration_and_strike_for_all_available_expirations_as_polars(ticker)
subset_strikes('AAPL', trading, expiration)
relevant_data = get_relevant_expirations_and_strikes('AAPL').select(['trading_day', 'expiration_day', 'strike','ticker'])
paired_expirations_and_strike = [row for row in relevant_data.iter_rows()]
# trading_expiration_pair = df.filter((pl.col('trading_day') == trading) & (pl.col('expiration_day') == expiration) & (pl.col('ticker') == ticker))
# trading_expiration_pair
relevant_data.sort('trading_day').write_csv('relevant_data.csv')

In [ ]:
get_unique_trading_days('AAPL').sort('trading_day')

In [ ]:
relevant_data[10:13]

In [ ]:
trading = datetime.datetime(2019, 9, 24)
relevant_data.filter(pl.col('trading_day') == trading)

In [ ]:
print(f"Number of Strikes subsetted for {ticker} = {len(relevant_data['strike'].unique())}")

In [ ]:
len(paired_expirations_and_strike)

In [ ]:
df = get_cleaned_options_data('AAPL')

In [ ]:
df['strike'].unique()

In [ ]:
df

In [ ]:
df.filter(pl.col('strike').is_between(187500,200000))

In [ ]:
221-95

In [ ]:
def get_ohlc_for_each_strike (df, strike, col_name, every='5m'):
    return df.filter(pl.col('strike') == strike).group_by_dynamic('timestamp', every=every).agg(
        pl.col('ticker').first().alias('ticker'),
        pl.col("strike").first().alias("strike"),
        pl.col("expiration").first().alias("expiration"),
        pl.col(col_name).first().alias("open"),
        pl.col(col_name).max().alias("high"),
        pl.col(col_name).min().alias("low"),
        pl.col(col_name).last().alias("close")
    )     
        

In [ ]:
get_ohlc_for_each_strike(df, 95000, "implied_vol")

In [ ]:
def get_ohlc_implied_volatility(ticker, every="5m"):
    get_rid_of_nans = pl.all_horizontal(pl.col(pl.Float32, pl.Float64).is_not_nan())
    df = get_cleaned_options_data(ticker).filter(
        get_rid_of_nans
    )  # Some 5 minute intervals contain NaNs
    ohlc_data = df.group_by_dynamic("timestamp", every=every).agg(
        pl.col('ticker').first().alias('ticker'),
        pl.col("strike").first().alias("strike"),
        pl.col("expiration").first().alias("expiration"),
        pl.col("implied_vol").first().alias("open"),
        pl.col("implied_vol").max().alias("high"),
        pl.col("implied_vol").min().alias("low"),
        pl.col("implied_vol").last().alias("close"),     
    )
    ohlc_data = ohlc_data.drop_nulls()
    return ohlc_data


In [ ]:
get_ohlc_data('AAPL', every='5m')